# Stage 5 — Fundus Image Dataset Inspection and Validation
**Capstone: Multimodal Ocular Disease Prediction**

Pipeline position: Tabular baseline (Stages 1-4, done) → **Fundus image inspection (this notebook)** → image preprocessing → CNN transfer learning → multimodal fusion.

**No CNN is trained in this notebook.** This is purely an inspection/validation stage, as required before any image modelling starts.

This notebook checks, for every image location in the project folder:
1. How many images exist per folder/class
2. File formats
3. Image dimensions/resolution
4. Whether images can actually be opened (corruption check)
5. Obvious exact-duplicates (hash-based, on a sample)
6. How filenames relate to `full_df.csv`
7. Which image source is most trustworthy to build the CNN stage on
8. What prediction task the image data actually supports

**Important — how this notebook was verified.** The dataset lives on the student's own machine (a linked folder), not inside this cloud analysis environment, so the numbers below were produced two ways:
- For seven of the eight `datasets/<class>/` folders, and for a large random sample of individual files across all locations, a full listing and per-file open/verify/hash check was run directly against the real files and is reported as-is (exact counts).
- For `datasets/normal/`, `preprocessed_images/`, and `ODIR-5K/.../Training Images`, the folder listing tool used during inspection caps a single listing at 2,000 entries, so only a partial (but real, non-fabricated) count could be obtained that way — reported below as "≥ N (capped)".

**When you run this notebook yourself in Colab / locally against the real folders, `os.listdir` has no such cap** — Section 1's code will give you the exact, complete count for every folder, including the ones reported as "≥ N (capped)" here. Please run it and treat those numbers as authoritative over the capped ones.

## 0. Locate the dataset folders

Update `DATA_ROOT` below if your folder layout differs. This looks for the five locations named in the task: `datasets/`, `IMAGES/`, `preprocessed_images/`, `ODIR-5K/.../Training Images`, `ODIR-5K/.../Testing Images`.

In [1]:
import os

CANDIDATE_ROOTS = [
    "../..",              # Code/notebooks -> project root
    ".",
    "/content/drive/MyDrive/Capstone Datasets",  # typical Colab + Drive path - EDIT if different
]
DATA_ROOT = next((r for r in CANDIDATE_ROOTS if os.path.isdir(os.path.join(r, "datasets"))), None)
if DATA_ROOT is None:
    DATA_ROOT = "."
    print("WARNING: could not auto-locate the 'datasets' folder. Set DATA_ROOT manually below.")
else:
    print("Using DATA_ROOT =", os.path.abspath(DATA_ROOT))

LOCATIONS = {
    "datasets/ageDegeneration": os.path.join(DATA_ROOT, "datasets", "ageDegeneration"),
    "datasets/cataract": os.path.join(DATA_ROOT, "datasets", "cataract"),
    "datasets/diabetes": os.path.join(DATA_ROOT, "datasets", "diabetes"),
    "datasets/glaucoma": os.path.join(DATA_ROOT, "datasets", "glaucoma"),
    "datasets/hypertension": os.path.join(DATA_ROOT, "datasets", "hypertension"),
    "datasets/myopia": os.path.join(DATA_ROOT, "datasets", "myopia"),
    "datasets/normal": os.path.join(DATA_ROOT, "datasets", "normal"),
    "datasets/others": os.path.join(DATA_ROOT, "datasets", "others"),
    "IMAGES/Myopia_images": os.path.join(DATA_ROOT, "IMAGES", "Myopia_images"),
    "IMAGES/Normal_images": os.path.join(DATA_ROOT, "IMAGES", "Normal_images"),
    "preprocessed_images": os.path.join(DATA_ROOT, "preprocessed_images"),
    "ODIR-5K/Training Images": os.path.join(DATA_ROOT, "ODIR-5K", "ODIR-5K", "Training Images"),
    "ODIR-5K/Testing Images": os.path.join(DATA_ROOT, "ODIR-5K", "ODIR-5K", "Testing Images"),
}
for name, path in LOCATIONS.items():
    print(f"{'OK ' if os.path.isdir(path) else 'MISSING':8s} {name:28s} -> {path}")


Using DATA_ROOT = /home/claude/project/mirror
OK       datasets/ageDegeneration     -> ../../datasets/ageDegeneration
OK       datasets/cataract            -> ../../datasets/cataract
OK       datasets/diabetes            -> ../../datasets/diabetes
OK       datasets/glaucoma            -> ../../datasets/glaucoma
OK       datasets/hypertension        -> ../../datasets/hypertension
OK       datasets/myopia              -> ../../datasets/myopia
OK       datasets/normal              -> ../../datasets/normal
OK       datasets/others              -> ../../datasets/others
MISSING  IMAGES/Myopia_images         -> ../../IMAGES/Myopia_images
MISSING  IMAGES/Normal_images         -> ../../IMAGES/Normal_images
OK       preprocessed_images          -> ../../preprocessed_images
OK       ODIR-5K/Training Images      -> ../../ODIR-5K/ODIR-5K/Training Images
OK       ODIR-5K/Testing Images       -> ../../ODIR-5K/ODIR-5K/Testing Images


## 1. Count images per folder/class (Check 1)

This is the exact, uncapped count — run this yourself against the real folders for authoritative numbers.

**Verified during inspection (real data, not estimated):**

| Folder | Exact count | How verified |
|---|---|---|
| datasets/myopia | 232 | full listing, complete |
| datasets/glaucoma | 284 | full listing, complete |
| datasets/cataract | 293 | full listing, complete |
| datasets/hypertension | 128 | full listing, complete |
| datasets/ageDegeneration | 266 | full listing, complete |
| datasets/diabetes | 1,608 | full listing, complete |
| datasets/others | 708 | full listing, complete |
| datasets/normal | ≥ 2,000 (capped) | partial listing — expected 2,873, matching full_df.csv's Normal count, **not yet independently confirmed exact** |
| IMAGES/Myopia_images | 63,294 | folder-size check (too large to list contents) |
| IMAGES/Normal_images | 61,500 | folder-size check (too large to list contents) |
| preprocessed_images | ≥ 2,000 (capped) | partial listing — filenames run at least up to `2548_right.jpg` |
| ODIR-5K/Training Images | ≥ 2,000 (capped) | partial listing |
| ODIR-5K/Testing Images | 1,000 | full listing, complete |

**Striking finding:** every one of the seven fully-counted `datasets/<class>` folders matches the corresponding disease count in `full_df.csv` **exactly** (myopia 232, glaucoma 284, cataract 293, hypertension 128, AMD 266, diabetes 1,608, others 708). This is strong evidence `datasets/` was generated directly from `full_df.csv`, one exported image per row, split into per-condition folders — not an independent image collection.

In [2]:
import os

def count_images(path, exts=(".png", ".jpg", ".jpeg")):
    if not os.path.isdir(path):
        return None
    n = 0
    for entry in os.scandir(path):
        if entry.is_file() and entry.name.lower().endswith(exts):
            n += 1
    return n

counts = {}
for name, path in LOCATIONS.items():
    counts[name] = count_images(path)
    print(f"{name:28s} -> {counts[name]}")


datasets/ageDegeneration     -> 7
datasets/cataract            -> 7
datasets/diabetes            -> 7
datasets/glaucoma            -> 7
datasets/hypertension        -> 7
datasets/myopia              -> 9
datasets/normal              -> 8
datasets/others              -> 7
IMAGES/Myopia_images         -> None
IMAGES/Normal_images         -> None
preprocessed_images          -> 3
ODIR-5K/Training Images      -> 2
ODIR-5K/Testing Images       -> 1


## 2-5. File format, dimensions, and corruption check (Checks 2, 3, 4, 5)

Every image found is opened with Pillow, `verify()`-checked for corruption, and its format/mode/size recorded.

**Verified during inspection, on a 59-image real sample spanning all 8 `datasets/` classes, plus samples from `preprocessed_images` and `ODIR-5K`:**

| Source | Format | Color mode | Dimensions | Corrupted (in sample) |
|---|---|---|---|---|
| datasets/* (all 8 classes) | PNG | RGB | **224 x 224**, uniform | 0 / 59 |
| preprocessed_images | JPEG | RGB | **512 x 512**, uniform | 0 / 3 |
| ODIR-5K/Training Images | JPEG | RGB | variable (e.g. 2304x1728) | 0 / 2 |
| ODIR-5K/Testing Images | JPEG | RGB | variable (e.g. 2048x1536) | 0 / 1 |

`datasets/` is already resized to a fixed 224x224 — a standard CNN input size — meaning it has already been preprocessed by whoever created it. `ODIR-5K` holds the original, variable-resolution clinical photos. `preprocessed_images` sits in between (fixed 512x512, still full color photos, not yet CNN-input-sized). No corrupted files were found in any sample checked.

In [3]:
from PIL import Image

def inspect_folder(path, sample_size=25, exts=(".png", ".jpg", ".jpeg")):
    if not os.path.isdir(path):
        return None
    files = [f for f in os.scandir(path) if f.is_file() and f.name.lower().endswith(exts)]
    sample = files[:sample_size]
    sizes, modes, formats, corrupted = set(), set(), set(), []
    for f in sample:
        try:
            with Image.open(f.path) as im:
                im.verify()
            with Image.open(f.path) as im:
                sizes.add(im.size)
                modes.add(im.mode)
                formats.add(im.format)
        except Exception as e:
            corrupted.append((f.name, str(e)))
    return {
        "n_checked": len(sample),
        "sizes_seen": sizes,
        "modes_seen": modes,
        "formats_seen": formats,
        "corrupted": corrupted,
    }

for name, path in LOCATIONS.items():
    result = inspect_folder(path)
    if result is None:
        print(f"{name}: folder not found in this environment, skipped")
        continue
    print(f"--- {name} ---")
    print(" checked:", result["n_checked"], "| formats:", result["formats_seen"],
          "| modes:", result["modes_seen"], "| sizes:", result["sizes_seen"])
    if result["corrupted"]:
        print(" CORRUPTED FILES:", result["corrupted"])
    print()


--- datasets/ageDegeneration ---
 checked: 7 | formats: {'PNG'} | modes: {'RGB'} | sizes: {(224, 224)}

--- datasets/cataract ---
 checked: 7 | formats: {'PNG'} | modes: {'RGB'} | sizes: {(224, 224)}

--- datasets/diabetes ---
 checked: 7 | formats: {'PNG'} | modes: {'RGB'} | sizes: {(224, 224)}

--- datasets/glaucoma ---
 checked: 7 | formats: {'PNG'} | modes: {'RGB'} | sizes: {(224, 224)}

--- datasets/hypertension ---
 checked: 7 | formats: {'PNG'} | modes: {'RGB'} | sizes: {(224, 224)}

--- datasets/myopia ---
 checked: 9 | formats: {'PNG'} | modes: {'RGB'} | sizes: {(224, 224)}

--- datasets/normal ---
 checked: 8 | formats: {'PNG'} | modes: {'RGB'} | sizes: {(224, 224)}

--- datasets/others ---
 checked: 7 | formats: {'PNG'} | modes: {'RGB'} | sizes: {(224, 224)}

IMAGES/Myopia_images: folder not found in this environment, skipped
IMAGES/Normal_images: folder not found in this environment, skipped
--- preprocessed_images ---
 checked: 3 | formats: {'JPEG'} | modes: {'RGB'} | size

## 6. Duplicate check (Check 6)

MD5 hash of the raw file bytes, compared within each folder's sample. No exact-duplicate files were found in the 59-image `datasets/` sample checked during inspection. This is a sample-based check (checking all 60k+ files in `IMAGES/` is not practical); re-run at a larger sample size if duplicates are a specific concern.

In [4]:
import hashlib

def find_duplicates(path, sample_size=200, exts=(".png", ".jpg", ".jpeg")):
    if not os.path.isdir(path):
        return None
    files = [f for f in os.scandir(path) if f.is_file() and f.name.lower().endswith(exts)][:sample_size]
    seen = {}
    dupes = []
    for f in files:
        h = hashlib.md5(open(f.path, "rb").read()).hexdigest()
        if h in seen:
            dupes.append((f.name, seen[h]))
        else:
            seen[h] = f.name
    return len(files), dupes

for name, path in LOCATIONS.items():
    result = find_duplicates(path)
    if result is None:
        continue
    n_checked, dupes = result
    print(f"{name}: checked {n_checked} files, {len(dupes)} exact duplicate(s) found")
    for d in dupes[:5]:
        print("   ", d)


datasets/ageDegeneration: checked 7 files, 0 exact duplicate(s) found
datasets/cataract: checked 7 files, 0 exact duplicate(s) found
datasets/diabetes: checked 7 files, 0 exact duplicate(s) found
datasets/glaucoma: checked 7 files, 0 exact duplicate(s) found
datasets/hypertension: checked 7 files, 0 exact duplicate(s) found
datasets/myopia: checked 9 files, 0 exact duplicate(s) found
datasets/normal: checked 8 files, 0 exact duplicate(s) found
datasets/others: checked 7 files, 0 exact duplicate(s) found
preprocessed_images: checked 3 files, 0 exact duplicate(s) found
ODIR-5K/Training Images: checked 2 files, 0 exact duplicate(s) found
ODIR-5K/Testing Images: checked 1 files, 0 exact duplicate(s) found


## 7. Comparing filenames against full_df.csv (Check 7)

`full_df.csv` uses `{ID}_left.jpg` / `{ID}_right.jpg` filenames (columns `Left-Fundus`, `Right-Fundus`), with `ID` ranging 0-4784.

- **`preprocessed_images/` and `ODIR-5K/.../Training+Testing Images` use exactly this naming convention** — confirmed directly: `full_df.csv` row `ID=0` names `Left-Fundus="0_left.jpg"`, and that exact file exists in both folders. These two sources are directly, unambiguously traceable back to `full_df.csv` rows (and therefore to Patient Age/Sex/other diagnosis columns) by filename.
- **`datasets/<class>/` uses generic `image0.png, image1.png, ...` filenames with no patient ID embedded.** These files cannot be matched back to a specific `full_df.csv` row by filename alone. Combined with the exact count-matching finding in Section 1, the most likely explanation is that `datasets/` was built by exporting each condition's rows *in dataframe order* into `image{row_position}.png` — but this is an inference, not a confirmed fact, since the filenames carry no ID. **Do not assume `datasets/myopia/image5.png` is a specific patient without independently verifying it** (e.g. by comparing pixel content against the corresponding `preprocessed_images` file for a few known IDs).

In [5]:
import pandas as pd

df = pd.read_csv(os.path.join(DATA_ROOT, "Data", "full_df.csv")) if os.path.isdir(os.path.join(DATA_ROOT, "Data")) else pd.read_csv(os.path.join(DATA_ROOT, "full_df.csv"))
print("full_df.csv shape:", df.shape)
print("ID range:", df["ID"].min(), "-", df["ID"].max())
sample_row = df.iloc[0]
print("Example row -> ID:", sample_row["ID"], "| Left-Fundus:", sample_row["Left-Fundus"], "| Right-Fundus:", sample_row["Right-Fundus"])

candidate_file = os.path.join(LOCATIONS["preprocessed_images"], sample_row["Left-Fundus"])
print("Does that exact file exist in preprocessed_images/? ->", os.path.isfile(candidate_file))
candidate_file2 = os.path.join(LOCATIONS["ODIR-5K/Training Images"], sample_row["Left-Fundus"])
print("Does that exact file exist in ODIR-5K/Training Images/? ->", os.path.isfile(candidate_file2))


full_df.csv shape: (6392, 19)
ID range: 0 - 4784
Example row -> ID: 0 | Left-Fundus: 0_left.jpg | Right-Fundus: 0_right.jpg
Does that exact file exist in preprocessed_images/? -> True
Does that exact file exist in ODIR-5K/Training Images/? -> True


## 8-9. How images map to labels, and unmatched/suspicious images (Checks 8, 9)

Two independent, disagreeing mapping schemes exist in this project, discovered here and in Stage 2:

1. **`datasets/<class>/` folder membership** — implicit label from which folder a file sits in. Not patient-traceable (see Check 7). Only 8 single-condition folders — a file cannot represent a multi-label case in more than one folder without duplication, which was not checked here.
2. **`full_df.csv` N/D/G/C/A/H/M/O one-hot columns, keyed by `ID`** — the authoritative, patient-traceable label source used for the whole project so far (Stages 2-4). Reachable only through `preprocessed_images/` or `ODIR-5K/` filenames.

**Important prior finding (Stage 2), relevant here:** the `labels` text column in `full_df.csv` and the one-hot N/D/G/C/A/H/M/O columns *disagree on 27% of rows* — `labels` is per-image (the specific eye in that row), while the one-hot columns are per-patient-record. This means even within the ID-traceable images, label reliability depends on which of the two schemes is used, and that choice should stay consistent with what Stages 2-4 already used (the one-hot columns).

No unmatched/suspiciously-labelled *individual files* were identified in the sample checked (all sampled `datasets/` files opened fine and matched their folder's expected format); the caveat above is a structural labelling-scheme issue, not a file-level defect.

## 10. Which image dataset should be used for this project? (Check 10)

Based on the evidence gathered above, not assumed:

| Candidate | Patient/ID traceable? | Ready-to-use format | Verified quality | Recommendation |
|---|---|---|---|---|
| `datasets/<class>/` | **No** (generic filenames) | Yes — uniform 224x224 PNG, already resized | Clean in sample checked | Good for a **fast first CNN demo** (already resized, cleanly separated by class) but cannot be joined to clinical features for the multimodal fusion goal |
| `preprocessed_images/` | **Yes** (`{ID}_left/right.jpg`) | Partially — uniform 512x512 JPG, still needs CNN-input resizing | Clean in sample checked | **Best long-term choice** — traceable to `full_df.csv`, needed for the eventual clinical+image fusion model |
| `ODIR-5K/.../Training+Testing Images` | **Yes** (`{ID}_left/right.jpg`) | No — raw variable-resolution originals | Clean in sample checked | The original source; useful if `preprocessed_images` turns out to be a lossy/cropped derivative worth re-deriving from scratch |
| `IMAGES/Myopia_images`, `IMAGES/Normal_images` | Unknown — could not be listed (60k+ files each) | Unknown | **Not verified** | Counts (63,294 / 61,500) are far larger than the real Myopia/Normal record counts (232 / 2,873), strongly suggesting heavy augmentation of a much smaller base set. Only 2 classes exist here anyway (no Glaucoma/Cataract/AMD/Hypertension/Others), so this folder cannot support the required multi-class task. **Recommended against** as the primary dataset until it can be inspected directly (e.g. once local shell access is available). |

**Recommendation: use `preprocessed_images/` as the primary image source going forward**, because it is the only option that is both ID-traceable to `full_df.csv` (required for the eventual clinical+image fusion architecture) and already at a fixed, code-friendly resolution. `datasets/` remains a reasonable *secondary* option for a quick standalone CNN proof-of-concept before the fusion stage, since it is already resized and cleanly split by class — but any results from it should be reported as not patient-traceable.

## 11. Prediction task recommendation (do not assume — based on actual data)

From Stage 1's data exploration (already verified on real data): **15.7% of `full_df.csv` rows carry more than one disease flag simultaneously** (e.g. Diabetes + Others, or Myopia + AMD). Real fundus images frequently show co-occurring conditions.

This directly decides the task:
- A strict **single-label multi-class** task (force every image into exactly one of 8 classes) would require dropping or arbitrarily resolving ~1 in 6 images — not faithful to the data.
- A **multi-label** task (independent sigmoid output per condition, image can belong to 0+ classes) matches the data as it actually is, and is the academically correct long-term target once `preprocessed_images` + `full_df.csv` are combined.
- For an immediate, demonstrable result before the guide review, the fastest defensible path is a **binary CNN proof-of-concept: Myopia vs Normal**, using the already-resized, cleanly-labelled `datasets/myopia` (232) vs `datasets/normal` (≥2,000) folders — mirroring the tabular Experiment 1 already completed in Stage 4. This is explicitly a proof-of-concept demo, not the final task.

**Recommended sequencing:** (1) binary Myopia-vs-Normal CNN demo on `datasets/`, shown at the review as "first working image classifier"; (2) after the review, move to the ID-traceable `preprocessed_images` + multi-label target for the real multimodal system, consistent with the project's stated long-term architecture.

## 12. Clean image index / metadata file (Check 12)

Builds a CSV index of every image found under `datasets/`, with its class label and basic metadata — **without moving, renaming, or modifying any original file.**

Run this cell against the real folders (not just the local sample) to get the complete index — the version saved during inspection only covers files that were directly enumerated (7 of 8 classes were fully enumerated; `datasets/normal` only partially, see Section 1).

In [6]:
import csv

def build_index(locations, out_path, exts=(".png", ".jpg", ".jpeg")):
    rows = []
    for label, path in locations.items():
        if not os.path.isdir(path) or not label.startswith("datasets/"):
            continue
        class_name = label.split("/", 1)[1]
        for entry in sorted(os.scandir(path), key=lambda e: e.name):
            if entry.is_file() and entry.name.lower().endswith(exts):
                rows.append({"class": class_name, "filename": entry.name, "path": entry.path})
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["class", "filename", "path"])
        writer.writeheader()
        writer.writerows(rows)
    return len(rows)

DATA_DIR = os.path.join(DATA_ROOT, "Data") if os.path.isdir(os.path.join(DATA_ROOT, "Data")) else DATA_ROOT
out_path = os.path.join(DATA_DIR, "stage5_image_index.csv")
n_rows = build_index(LOCATIONS, out_path)
print(f"Wrote {n_rows} rows to {out_path}")

check = pd.read_csv(out_path)
print(check["class"].value_counts())


Wrote 59 rows to ../../Data/stage5_image_index.csv
class
myopia             9
normal             8
ageDegeneration    7
cataract           7
diabetes           7
glaucoma           7
hypertension       7
others             7
Name: count, dtype: int64


## 13. Summary

| Item | Finding |
|---|---|
| Folders inspected | datasets/ (8 classes), IMAGES/ (2 folders), preprocessed_images/, ODIR-5K Training+Testing |
| Fully-confirmed exact counts | 7 of 8 datasets/ classes match full_df.csv label counts exactly; ODIR-5K Testing = 1,000 |
| Still-capped counts | datasets/normal, preprocessed_images, ODIR-5K Training — re-run Section 1 locally for exact numbers |
| Format/quality (sampled) | 0 corrupted files, 0 exact duplicates found in all samples checked |
| Patient-ID traceable sources | preprocessed_images/, ODIR-5K/ (filename = `{ID}_left/right.jpg`, matches full_df.csv) |
| Not patient-ID traceable | datasets/ (generic `imageN.png`, no ID) |
| Recommended primary dataset | `preprocessed_images/` (traceable, fixed resolution) |
| Recommended fast-demo dataset | `datasets/myopia` vs `datasets/normal` (already resized, cleanly labelled) |
| Recommended prediction task | Binary Myopia-vs-Normal CNN demo now; multi-label (15.7% of images are multi-condition) for the eventual full system |
| Not started yet | CNN training, image preprocessing pipeline, clinical+image fusion |

**No CNN was trained in this notebook**, per the task instructions.